# Build Bronze Maintenance Events

This notebook promotes JetOps maintenance events from the raw Event Hubs Capture path into a Bronze Delta dataset.

Execution behavior:
- Databricks: reads raw Avro, parses the event body, and writes Delta to the bronze container.
- Local VS Code notebook: previews the transformed Bronze rows from the latest captured Avro file without writing Delta.

Defaults target the dev environment. Override them with `JETOPS_*` environment variables when needed.

In [ ]:
import os

storage_account = os.getenv("JETOPS_STORAGE_ACCOUNT", "stherbalifedev001")
raw_container = os.getenv("JETOPS_RAW_CONTAINER", "raw")
bronze_container = os.getenv("JETOPS_BRONZE_CONTAINER", "bronze")
secret_scope = os.getenv("JETOPS_SECRET_SCOPE", "herbalife-storage")
raw_secret_key = os.getenv("JETOPS_SECRET_KEY", "raw-sas-token")
bronze_secret_key = os.getenv("JETOPS_BRONZE_SECRET_KEY", raw_secret_key)
storage_account_key_secret = os.getenv("JETOPS_STORAGE_ACCOUNT_KEY_SECRET", "storage-account-key")
storage_auth_mode = os.getenv("JETOPS_STORAGE_AUTH_MODE", "account_key")
eventhub_namespace = os.getenv("JETOPS_EVENTHUB_NAMESPACE", "evh-herbalife-dev")
eventhub_name = os.getenv("JETOPS_EVENTHUB_NAME", "jetops-maintenance-events-dev")
capture_root = os.getenv("JETOPS_CAPTURE_ROOT", "jetops-maintenance")
resource_group = os.getenv("JETOPS_RESOURCE_GROUP", "rg-herbalife-dev-core")
bronze_dataset = os.getenv("JETOPS_BRONZE_DATASET", "jetops/maintenance_events")
write_mode = os.getenv("JETOPS_BRONZE_WRITE_MODE", "overwrite")
az_cli = os.getenv("AZURE_CLI_PATH", r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd")

raw_capture_path = (
    f"wasbs://{raw_container}@{storage_account}.blob.core.windows.net/"
    f"{capture_root}/{eventhub_namespace}/{eventhub_name}/*/*/*/*/*/*/*"
)
bronze_delta_path = f"wasbs://{bronze_container}@{storage_account}.blob.core.windows.net/{bronze_dataset}"

is_databricks = "dbutils" in globals() and "spark" in globals()
print(f"Execution mode: {'databricks' if is_databricks else 'local'}")
print(f"Raw capture path: {raw_capture_path}")
print(f"Bronze Delta path: {bronze_delta_path}")
print(f"Write mode: {write_mode}")
print(f"Databricks storage auth mode: {storage_auth_mode}")

if is_databricks:
    if storage_auth_mode == "sas":
        raw_sas_token = dbutils.secrets.get(scope=secret_scope, key=raw_secret_key)
        bronze_sas_token = dbutils.secrets.get(scope=secret_scope, key=bronze_secret_key)
        spark.conf.set(
            f"fs.azure.sas.{raw_container}.{storage_account}.blob.core.windows.net",
            raw_sas_token,
        )
        spark.conf.set(
            f"fs.azure.sas.{bronze_container}.{storage_account}.blob.core.windows.net",
            bronze_sas_token,
        )
    else:
        storage_account_key = dbutils.secrets.get(scope=secret_scope, key=storage_account_key_secret)
        spark.conf.set(
            f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
            storage_account_key,
        )
else:
    print("Local mode will preview transformed Bronze rows from the latest captured Avro file.")

Execution mode: local
Raw capture path: wasbs://raw@stherbalifedev001.blob.core.windows.net/jetops-maintenance/evh-herbalife-dev/jetops-maintenance-events-dev/*/*/*/*/*/*/*
Bronze Delta path: wasbs://bronze@stherbalifedev001.blob.core.windows.net/jetops/maintenance_events
Write mode: overwrite
Local mode will preview transformed Bronze rows from the latest captured Avro file.


In [ ]:
import json
import subprocess
import tempfile
from datetime import UTC, datetime
from pathlib import Path

maintenance_schema = """
event_type STRING,
event_id STRING,
event_time_utc STRING,
tail_number STRING,
aircraft_model STRING,
maintenance_log_id INT,
work_order_id STRING,
status STRING,
maintenance_type STRING,
component STRING,
fault_code STRING,
severity STRING,
priority STRING,
part_hours DOUBLE,
estimated_downtime_hours DOUBLE,
labor_hours_estimate DOUBLE,
inspection_date STRING,
technician_id STRING,
hangar STRING,
airport_code STRING,
operator_name STRING,
route_segment STRING,
maintenance_station STRING,
maintenance_category STRING,
dispatch_impact STRING,
requires_parts INT,
part_order_status STRING,
parts_eta_hours DOUBLE,
repeat_issue_flag INT,
reported_by STRING,
event_source_system STRING,
recommended_action STRING,
details STRING,
ingestion_source STRING,
schema_version STRING
"""

if is_databricks:
    from pyspark.sql.functions import col, current_timestamp, decode, from_json, input_file_name, to_date, to_timestamp, to_json

    raw_df = spark.read.format("avro").load(raw_capture_path)
    bronze_df = (
        raw_df
        .withColumn("body_text", decode(col("Body"), "UTF-8"))
        .withColumn("event", from_json(col("body_text"), maintenance_schema))
        .select(
            col("SequenceNumber").alias("sequence_number"),
            col("Offset").alias("offset"),
            to_timestamp(col("EnqueuedTimeUtc")).alias("enqueued_time_utc"),
            to_json(col("SystemProperties")).alias("system_properties_json"),
            to_json(col("Properties")).alias("properties_json"),
            col("body_text").alias("body_json"),
            col("event.*"),
            input_file_name().alias("source_file"),
            current_timestamp().alias("bronze_loaded_at")
        )
        .withColumn("event_timestamp", to_timestamp(col("event_time_utc")))
        .withColumn("event_date", to_date(col("event_timestamp")))
    )

    writer = (
        bronze_df.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
    )
    if write_mode != "overwrite":
        writer = writer.option("mergeSchema", "true")

    (
        writer
        .partitionBy("event_date")
        .save(bronze_delta_path)
    )

    print(f"Wrote Bronze Delta dataset to {bronze_delta_path}")
    print(f"Rows written: {bronze_df.count()}")
else:
    try:
        from fastavro import reader
    except ImportError as exc:
        raise ImportError(
            "Local mode requires fastavro in the notebook kernel. Install it before running this cell."
        ) from exc

    capture_prefix = f"{capture_root}/{eventhub_namespace}/{eventhub_name}"
    account_key = subprocess.check_output(
        [
            az_cli,
            "storage",
            "account",
            "keys",
            "list",
            "--resource-group",
            resource_group,
            "--account-name",
            storage_account,
            "--query",
            "[0].value",
            "-o",
            "tsv",
        ],
        text=True,
    ).strip()

    file_list = subprocess.check_output(
        [
            az_cli,
            "storage",
            "fs",
            "file",
            "list",
            "--account-name",
            storage_account,
            "--account-key",
            account_key,
            "--file-system",
            raw_container,
            "--path",
            capture_prefix,
            "--exclude-dir",
            "-o",
            "json",
        ],
        text=True,
    )
    files = json.loads(file_list)
    avro_files = sorted(file_info["name"] for file_info in files if file_info["name"].endswith(".avro"))
    if not avro_files:
        raise FileNotFoundError(f"No Avro capture files found under {capture_prefix}")

    latest_file = avro_files[-1]
    bronze_loaded_at = datetime.now(UTC).isoformat().replace("+00:00", "Z")

    with tempfile.TemporaryDirectory() as temp_dir:
        local_file = Path(temp_dir) / Path(latest_file).name
        subprocess.run(
            [
                az_cli,
                "storage",
                "fs",
                "file",
                "download",
                "--account-name",
                storage_account,
                "--account-key",
                account_key,
                "--file-system",
                raw_container,
                "--path",
                latest_file,
                "--destination",
                str(local_file),
                "--overwrite",
                "true",
            ],
            check=True,
            capture_output=True,
            text=True,
        )

        with local_file.open("rb") as handle:
            records = list(reader(handle))

    bronze_preview = []
    for record in records[:5]:
        body_json = record["Body"].decode("utf-8")
        payload = json.loads(body_json)
        event_timestamp = payload.get("event_time_utc")
        bronze_preview.append({
            "sequence_number": record.get("SequenceNumber"),
            "offset": record.get("Offset"),
            "enqueued_time_utc": record.get("EnqueuedTimeUtc"),
            "system_properties_json": json.dumps(record.get("SystemProperties", {})),
            "properties_json": json.dumps(record.get("Properties", {})),
            "body_json": body_json,
            **payload,
            "source_file": latest_file,
            "bronze_loaded_at": bronze_loaded_at,
            "event_timestamp": event_timestamp,
            "event_date": (event_timestamp or "")[:10],
        })

    print(f"Latest raw file: {latest_file}")
    print(f"Records in latest raw file: {len(records)}")
    print("Previewing the first 5 transformed Bronze rows.")
    bronze_preview

Latest raw file: jetops-maintenance/evh-herbalife-dev/jetops-maintenance-events-dev/1/2026/04/05/03/49/50.avro
Records in latest raw file: 12618
Previewing the first 5 transformed Bronze rows.


In [3]:
if is_databricks:
    bronze_delta_df = spark.read.format("delta").load(bronze_delta_path)
    print(f"Bronze row count: {bronze_delta_df.count()}")
    display(bronze_delta_df.orderBy(col("event_timestamp").desc()))
else:
    print("Local mode does not write Delta. Use the Bronze preview from Cell 3 to validate the transformation shape.")

Local mode does not write Delta. Use the Bronze preview from Cell 3 to validate the transformation shape.
